# B1 — Kiểm tra Môi trường `.venv`, GPU RTX 5070 & Dữ liệu gốc

**Mục tiêu của Bước 1 (B1):**
1. Xác nhận JupyterLab đang chạy đúng môi trường ảo `.venv`.
2. Kiểm tra PyTorch + CUDA + **GPU RTX 5070 (12 GB VRAM)** + khả năng hỗ trợ **`bf16`** (ổn định & nhanh hơn `fp16` trên kiến trúc Blackwell).
3. Kiểm tra toàn bộ thư viện cần thiết trong `requirements.txt` và in lệnh cài nếu còn thiếu.
4. Kiểm tra 3 file dữ liệu gốc trong `data/` và khởi tạo sẵn cây thư mục lưu trữ biểu đồ báo cáo tại `workspace_outputs/figures/`.

In [1]:
# Bước 1.1: Thiết lập đường dẫn dự án và kiểm tra môi trường Python / .venv
import os
import sys
import platform
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    PipelineConfig,
    ensure_directories,
    RAW_TRAIN_PATH,
    RAW_TEST_PATH,
    RAW_UNLABELED_PATH,
    FIGURE_SUBDIRS,
)

ensure_directories()

print("=== THÔNG TIN MÔI TRƯỜNG PYTHON ===")
print(f"- Thư mục dự án   : {PROJECT_ROOT}")
print(f"- Python thực thi : {sys.executable}")
print(f"- Phiên bản Python: {platform.python_version()}")
print(f"- Hệ điều hành    : {platform.platform()}")
print(f"- Số luồng CPU    : {os.cpu_count()}")
if ".venv" in sys.executable.lower():
    print("[OK] Đang chạy đúng trong môi trường ảo .venv!")
else:
    print("[Cảnh báo] Kernel hiện tại KHÔNG nằm trong .venv — hãy chọn lại kernel .venv trên JupyterLab.")

=== THÔNG TIN MÔI TRƯỜNG PYTHON ===
- Thư mục dự án   : D:\Document 2025 Backup\XuLyNgonNguTuNhien\Vi-ESG-Bert-LuuTuanTrinh
- Python thực thi : D:\Document 2025 Backup\XuLyNgonNguTuNhien\Vi-ESG-Bert-LuuTuanTrinh\.venv\Scripts\python.exe
- Phiên bản Python: 3.14.7
- Hệ điều hành    : Windows-11-10.0.26200-SP0
- Số luồng CPU    : 20
[OK] Đang chạy đúng trong môi trường ảo .venv!


In [2]:
# Bước 1.2: Kiểm tra PyTorch, CUDA, GPU RTX 5070 và kiểm thử BF16
try:
    import torch
    print("=== KIỂM TRA PYTORCH & GPU ===")
    print(f"- PyTorch version : {torch.__version__}")
    print(f"- CUDA built-in   : {torch.version.cuda}")
    print(f"- CUDA available  : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        vram_gb = props.total_memory / (1024 ** 3)
        bf16_ok = torch.cuda.is_bf16_supported()
        print(f"- Tên GPU         : {gpu_name}")
        print(f"- Tổng VRAM       : {vram_gb:.2f} GB")
        print(f"- Compute Cap.    : {props.major}.{props.minor}")
        print(f"- Hỗ trợ BF16     : {bf16_ok}")

        # Thử phép nhân ma trận nhỏ trên GPU với bfloat16 để chắc chắn kernel chạy mượt
        dtype = torch.bfloat16 if bf16_ok else torch.float16
        x = torch.randn((512, 512), device="cuda", dtype=dtype)
        y = torch.matmul(x, x)
        torch.cuda.synchronize()
        del x, y
        torch.cuda.empty_cache()
        print(f"[OK] Kiểm thử tính toán GPU ({dtype}) thành công!")
    else:
        print("[Cảnh báo] PyTorch chưa nhận GPU CUDA!")
        print("Với RTX 5070 (Blackwell), hãy chạy lệnh sau trong terminal .venv:")
        print("  pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128")
except ImportError:
    print("[Thiếu] Chưa cài PyTorch!")
    print("Hãy chạy lệnh sau trong terminal .venv:")
    print("  pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128")

=== KIỂM TRA PYTORCH & GPU ===
- PyTorch version : 2.11.0+cu128
- CUDA built-in   : 12.8
- CUDA available  : True
- Tên GPU         : NVIDIA GeForce RTX 5070
- Tổng VRAM       : 11.94 GB
- Compute Cap.    : 12.0
- Hỗ trợ BF16     : True
[OK] Kiểm thử tính toán GPU (torch.bfloat16) thành công!


In [3]:
# Bước 1.3: Kiểm tra danh sách thư viện cần thiết trong requirements.txt
import importlib

required_packages = [
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("datasets", "datasets"),
    ("safetensors", "safetensors"),
    ("sentencepiece", "sentencepiece"),
    ("pyvi", "pyvi"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("scipy", "scipy"),
    ("pyarrow", "pyarrow"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("wordcloud", "wordcloud"),
    ("ipywidgets", "ipywidgets"),
    ("tqdm", "tqdm"),
    ("psutil", "psutil"),
]

missing = []
print("=== KIỂM TRA THƯ VIỆN ===")
for mod_name, pip_name in required_packages:
    try:
        mod = importlib.import_module(mod_name)
        ver = getattr(mod, "__version__", "OK")
        print(f"  [OK] {pip_name:<16} -> phiên bản {ver}")
    except ImportError:
        missing.append(pip_name)
        print(f"  [THIẾU] {pip_name}")

if missing:
    print("\nCần cài thêm các gói còn thiếu bằng lệnh:")
    print("  pip install -r requirements.txt")
else:
    print("\n[HOÀN HẢO] Toàn bộ thư viện đã được cài đầy đủ!")

=== KIỂM TRA THƯ VIỆN ===
  [OK] transformers     -> phiên bản 5.24.0
  [OK] accelerate       -> phiên bản 1.16.0
  [OK] datasets         -> phiên bản 5.2.0
  [OK] safetensors      -> phiên bản 0.8.0
  [OK] sentencepiece    -> phiên bản 0.2.3
  [OK] pyvi             -> phiên bản OK
  [OK] numpy            -> phiên bản 2.5.4
  [OK] pandas           -> phiên bản 3.1.1
  [OK] scikit-learn     -> phiên bản 1.9.1
  [OK] scipy            -> phiên bản 1.18.1
  [OK] pyarrow          -> phiên bản 26.0.1
  [OK] matplotlib       -> phiên bản 3.11.2
  [OK] seaborn          -> phiên bản 0.13.2
  [OK] wordcloud        -> phiên bản 1.9.6
  [OK] ipywidgets       -> phiên bản 8.1.9
  [OK] tqdm             -> phiên bản 4.70.1
  [OK] psutil           -> phiên bản 7.2.2

[HOÀN HẢO] Toàn bộ thư viện đã được cài đầy đủ!


In [4]:
# Bước 1.4: Kiểm tra 3 file dữ liệu gốc, thư mục lưu biểu đồ báo cáo & công tắc hệ thống
print("=== KIỂM TRA DỮ LIỆU GỐC ===")
for label, path in [
    ("Train gốc", RAW_TRAIN_PATH),
    ("Test gốc (Hold-out)", RAW_TEST_PATH),
    ("Unlabeled gốc", RAW_UNLABELED_PATH),
]:
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"  [OK] {label:<22}: {path.name:<20} ({size_mb:.2f} MB)")
    else:
        print(f"  [LỖI] Không tìm thấy {label}: {path}")

print("\n=== THƯ MỤC LƯU TRỮ BIỂU ĐỒ PHỤC VỤ BÁO CÁO ===")
for step, fig_dir in FIGURE_SUBDIRS.items():
    print(f"  - {step}: {fig_dir}")

print()
cfg = PipelineConfig(hardware_mode="balanced", model_preset="full_7")
print(cfg.summary())

=== KIỂM TRA DỮ LIỆU GỐC ===
  [OK] Train gốc             : train.csv            (5.98 MB)
  [OK] Test gốc (Hold-out)   : test.csv             (0.77 MB)
  [OK] Unlabeled gốc         : overall_data.csv     (15.21 MB)

=== THƯ MỤC LƯU TRỮ BIỂU ĐỒ PHỤC VỤ BÁO CÁO ===
  - B2: D:\Document 2025 Backup\XuLyNgonNguTuNhien\Vi-ESG-Bert-LuuTuanTrinh\workspace_outputs\figures\B2_eda
  - B3: D:\Document 2025 Backup\XuLyNgonNguTuNhien\Vi-ESG-Bert-LuuTuanTrinh\workspace_outputs\figures\B3_baseline_run1
  - B4: D:\Document 2025 Backup\XuLyNgonNguTuNhien\Vi-ESG-Bert-LuuTuanTrinh\workspace_outputs\figures\B4_pseudo_round1
  - B5: D:\Document 2025 Backup\XuLyNgonNguTuNhien\Vi-ESG-Bert-LuuTuanTrinh\workspace_outputs\figures\B5_run2
  - B6: D:\Document 2025 Backup\XuLyNgonNguTuNhien\Vi-ESG-Bert-LuuTuanTrinh\workspace_outputs\figures\B6_pseudo_round2
  - B7: D:\Document 2025 Backup\XuLyNgonNguTuNhien\Vi-ESG-Bert-LuuTuanTrinh\workspace_outputs\figures\B7_run3_ensemble
  - B8: D:\Document 2025 Backup\XuLyNgon